# Training Model Prediksi Penyakit Ginjal Kronis

Notebook ini khusus untuk proses training model. Hasil akhirnya disimpan ke `model_artifacts.joblib`, lalu file tersebut dipakai oleh `app.py`.

## 1. Import Library dan Konfigurasi

Bagian ini menyiapkan library yang dibutuhkan, lokasi dataset, lokasi file model yang akan disimpan, daftar kolom fitur, dan nama label fitur untuk tampilan visualisasi.

In [1]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell ini digunakan untuk import library dan mengatur konfigurasi utama training.
# Semua nama kolom fitur, target, dan lokasi file disiapkan di sini.

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, auc, confusion_matrix, f1_score, precision_score, recall_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

BASE_DIR = Path.cwd()
DATASET_FILE = BASE_DIR / "kidney_disease.csv"
MODEL_FILE = BASE_DIR / "model_artifacts.joblib"

TARGET_COLUMN = "classification"
NUMERIC_COLUMNS = [
    "age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wc", "rc",
]
CATEGORICAL_COLUMNS = [
    "rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane",
]
FIELD_LABELS = {
    "age": "Umur", "bp": "Tekanan darah", "sg": "Specific gravity",
    "al": "Albumin", "su": "Sugar", "rbc": "Red blood cells",
    "pc": "Pus cell", "pcc": "Pus cell clumps", "ba": "Bacteria",
    "bgr": "Blood glucose random", "bu": "Blood urea",
    "sc": "Serum creatinine", "sod": "Sodium", "pot": "Potassium",
    "hemo": "Hemoglobin", "pcv": "Packed cell volume",
    "wc": "White blood cell count", "rc": "Red blood cell count",
    "htn": "Hypertension", "dm": "Diabetes mellitus",
    "cad": "Coronary artery disease", "appet": "Appetite",
    "pe": "Pedal edema", "ane": "Anemia",
}

FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
CLASS_LABELS = {0: "Not CKD", 1: "CKD"}
RANDOM_STATE = 42

## 2. Fungsi Cleaning Data dan Pipeline Model

Bagian ini membuat fungsi untuk membersihkan isi dataset, mengubah kolom numerik menjadi angka, menyamakan format label target, dan membangun pipeline Random Forest. Pipeline berisi imputer untuk data kosong, StandardScaler untuk normalisasi, one-hot encoder untuk data kategori, SMOTE untuk menyeimbangkan data latih, dan model klasifikasi.

Catatan penting: SMOTE dipasang memakai `imblearn.pipeline.Pipeline` sehingga **hanya aktif pada saat `fit` (data training)**. Saat `predict` pada data uji, SMOTE otomatis dilewati, jadi data uji tetap asli dan tidak terjadi kebocoran data.

In [3]:
# Cell ini berisi fungsi bantuan untuk membersihkan data dan membuat pipeline model.
# Pipeline memastikan preprocessing, SMOTE, dan model berjalan dalam satu alur yang rapi.

#Data cleaning 
def clean_text(value):
    if pd.isna(value):
        return np.nan
    cleaned = str(value).strip().lower()
    return np.nan if cleaned in {"", "?", "nan"} else cleaned


def load_and_clean_data(path):
    df = pd.read_csv(path)
    df.columns = [str(col).strip().lower() for col in df.columns]

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].map(clean_text)

    #Transformasi Tipe Data
    for col in NUMERIC_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    #Normalisasi Label Target
    df[TARGET_COLUMN] = (
        df[TARGET_COLUMN]
        .map(clean_text)
        .replace({"ckd\t": "ckd", "not ckd": "notckd", "not_ckd": "notckd"})
    )
    return df


def create_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

#Data numerik/Fungsinya adalah mengisi nilai kosong (missing value) pada fitur numerik menggunakan nilai median,
#lalu dinormalisasi menggunakan StandardScaler. 
def create_numeric_pipeline():
    return Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])


def create_preprocessor():
    numeric_pipeline = create_numeric_pipeline()

    #Data kategorikal/berfungsi mengisi data kosong dengan nilai yang paling sering muncul (most frequent) dan mengubah data kategorikal menjadi data numerik menggunakan One-Hot Encoding.
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", create_one_hot_encoder()),
        ]
    )

    #Menggabungkan preprocessing(fitur numerik dan fitur kategorikal)
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_COLUMNS),
            ("cat", categorical_pipeline, CATEGORICAL_COLUMNS),
        ]
    )

#Membuat fungsi pipeline yaitu menggabungkan proses preprocessing, SMOTE, dan algoritma klasifikasi dalam satu alur kerja.
def create_model_pipeline(model_type="random_forest", n_estimators=250, use_smote=True):
    if model_type == "random_forest":
        #Membangun Model Random Forest
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            min_samples_leaf=2,
        )
    elif model_type == "svm":
        model = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)
    elif model_type == "naive_bayes":
        model = GaussianNB()
    else:
        raise ValueError(f"Model tidak dikenal: {model_type}")

    #Menggabungkan Preprocessing + SMOTE + Model
    #SMOTE dari imblearn hanya dijalankan saat fit (data training), tidak saat predict (data uji).
    steps = [("preprocessor", create_preprocessor())]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)))
    steps.append(("model", model))

    return ImbPipeline(steps=steps)

## 3. Load Dataset dan Split Data

Bagian ini membaca dataset, memilih fitur dan target, lalu membagi data menjadi data latih (80%) dan data uji (20%) secara stratified agar proporsi kelas tetap seimbang di kedua bagian.

In [4]:
# Cell ini membaca dataset, menyiapkan fitur-target, dan membagi data menjadi data latih dan data uji.

#Membaca Dataset dan Menyiapkan Data
df = load_and_clean_data(DATASET_FILE)
model_df = df[FEATURES + [TARGET_COLUMN]].copy().dropna(subset=[TARGET_COLUMN])
model_df[TARGET_COLUMN] = model_df[TARGET_COLUMN].map({"notckd": 0, "ckd": 1})

x = model_df[FEATURES]
y = model_df[TARGET_COLUMN]

#Split Data
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Jumlah seluruh data : {len(x)} pasien")
print(f"Jumlah data training : {len(x_train)} pasien")
print(f"Jumlah data testing  : {len(x_test)} pasien")

Jumlah seluruh data : 400 pasien
Jumlah data training : 320 pasien
Jumlah data testing  : 80 pasien


## 4. Hasil Normalisasi Data Menggunakan StandardScaler

Bagian ini menampilkan contoh hasil proses normalisasi pada **5 pasien pertama dari dataset**. Nilai kosong pada fitur numerik lebih dulu diisi dengan median, kemudian dinormalisasi dengan StandardScaler sehingga setiap fitur memiliki rata-rata 0 dan standar deviasi 1.

In [5]:
# Cell ini menampilkan perbandingan data pasien sebelum dan sesudah proses normalisasi StandardScaler.
# Scaler di-fit pada data training saja, lalu dipakai untuk mentransformasi 5 pasien pertama dataset.

numeric_pipeline_demo = create_numeric_pipeline()
numeric_pipeline_demo.fit(x_train[NUMERIC_COLUMNS])

sample_before = x[NUMERIC_COLUMNS].head(5).copy()
sample_after = pd.DataFrame(
    numeric_pipeline_demo.transform(sample_before),
    columns=NUMERIC_COLUMNS,
    index=sample_before.index,
).round(4)

sample_before.index = [f"Pasien {i + 1}" for i in range(len(sample_before))]
sample_after.index = [f"Pasien {i + 1}" for i in range(len(sample_after))]

print("Data Pasien Sebelum Proses Normalisasi")
display(sample_before)

print("\nData Pasien Sesudah Proses Normalisasi Menggunakan Metode StandardScaler")
display(sample_after)

Data Pasien Sebelum Proses Normalisasi


,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
Pasien 1,48.0,80.0,1.020,1.0,0.0,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2
Pasien 2,7.0,50.0,1.020,4.0,0.0,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN
Pasien 3,62.0,80.0,1.010,2.0,3.0,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN
Pasien 4,48.0,70.0,1.005,4.0,0.0,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9
Pasien 5,51.0,80.0,1.010,2.0,0.0,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6



Data Pasien Sesudah Proses Normalisasi Menggunakan Metode StandardScaler


,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
Pasien 1,-0.1867,0.2244,0.4270,0.0941,-0.3477,-0.2955,-0.4259,-0.3136,0.0755,-0.0607,1.0396,0.6248,-0.1980,0.5460
Pasien 2,-2.6099,-1.9296,0.4270,2.4116,-0.3477,-0.3025,-0.7994,-0.3814,0.0755,-0.0607,-0.4454,-0.1125,-0.8781,0.0673
Pasien 3,0.6407,0.2244,-1.4447,0.8666,2.7427,3.9267,-0.0732,-0.2120,0.0755,-0.0607,-1.0611,-0.9727,-0.3113,0.0673
Pasien 4,-0.1867,-0.4936,-2.3805,2.4116,-0.3477,-0.3514,-0.0110,0.1269,-2.7065,-2.7907,-0.4816,-0.8498,-0.6136,-1.0097
Pasien 5,-0.0094,0.2244,-1.4447,0.8666,-0.3477,-0.5052,-0.6334,-0.2797,0.0755,-0.0607,-0.3367,-0.4812,-0.3869,-0.1720


## 5. Penerapan SMOTE pada Data Training

Jumlah pasien CKD dan Not CKD pada dataset tidak seimbang. SMOTE (Synthetic Minority Over-sampling Technique) dipakai untuk membuat data sintetis pada kelas minoritas sehingga jumlah kedua kelas menjadi sama.

SMOTE **hanya diterapkan pada data training**. Data uji dibiarkan apa adanya supaya hasil evaluasi tetap mencerminkan kondisi data nyata.

In [6]:
# Cell ini menampilkan jumlah data training sebelum dan sesudah dilakukan SMOTE.
# SMOTE dijalankan setelah preprocessing karena SMOTE membutuhkan data dalam bentuk numerik.

preprocessor_demo = create_preprocessor()
x_train_prep = preprocessor_demo.fit_transform(x_train)

smote_demo = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
x_train_smote, y_train_smote = smote_demo.fit_resample(x_train_prep, y_train)


def ringkas_distribusi(y_series, jumlah_baris):
    counts = pd.Series(y_series).value_counts().sort_index()
    return pd.DataFrame(
        {
            "Kelas": [CLASS_LABELS[k] for k in counts.index],
            "Jumlah Data": counts.values,
            "Persentase (%)": (counts.values / counts.values.sum() * 100).round(2),
        }
    ).assign(**{"Total Data Training": jumlah_baris})


print("Data training sebelum di lakukan SMOTE")
display(ringkas_distribusi(y_train, len(y_train)))

print("\nData training sesudah dilakukan SMOTE")
display(ringkas_distribusi(y_train_smote, len(y_train_smote)))

print(f"\nJumlah data training bertambah dari {len(y_train)} menjadi {len(y_train_smote)} baris.")
print(f"Jumlah data testing tetap {len(y_test)} baris (tidak dikenai SMOTE).")

Data training sebelum di lakukan SMOTE


,Kelas,Jumlah Data,Persentase (%),Total Data Training
0,Not CKD,120,37.5,320
1,CKD,200,62.5,320



Data training sesudah dilakukan SMOTE


,Kelas,Jumlah Data,Persentase (%),Total Data Training
0,Not CKD,200,50.0,400
1,CKD,200,50.0,400



Jumlah data training bertambah dari 320 menjadi 400 baris.
Jumlah data testing tetap 80 baris (tidak dikenai SMOTE).


## 6. Training Model, Evaluasi, dan Perbandingan Model

Bagian ini melatih model evaluasi memakai data latih (dengan SMOTE di dalam pipeline), lalu menghitung akurasi training, akurasi testing, precision, recall, f1-score, confusion matrix, dan ROC Curve. Tiga algoritma dibandingkan: Naive Bayes, SVM, dan Random Forest.

In [7]:
# Cell ini melatih model dan menghitung accuracy, precision, recall, f1-score,
# confusion matrix, ROC Curve, serta perbandingan antar model.

#Proses Training Model/Pelatihan model menggunakan data latih (SMOTE otomatis dijalankan di dalam pipeline)
eval_pipeline = create_model_pipeline(model_type="random_forest", n_estimators=250)
eval_pipeline.fit(x_train, y_train)

#Prediksi
y_train_pred = eval_pipeline.predict(x_train)
y_pred = eval_pipeline.predict(x_test)
y_proba = eval_pipeline.predict_proba(x_test)[:, 1]

#Perbandingan Model
model_candidates = {
    "Naive Bayes": create_model_pipeline(model_type="naive_bayes"),
    "SVM": create_model_pipeline(model_type="svm"),
    "Random Forest": create_model_pipeline(model_type="random_forest", n_estimators=250),
}

roc_rows = []
model_comparison_rows = []
#Pembuatan ROC Curve
for model_name, candidate_pipeline in model_candidates.items():
    candidate_pipeline.fit(x_train, y_train)
    candidate_pred = candidate_pipeline.predict(x_test)
    candidate_proba = candidate_pipeline.predict_proba(x_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, candidate_proba, pos_label=1)
    model_auc = auc(fpr, tpr)

    #Perbandingan Antar Model
    model_comparison_rows.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, candidate_pred),
            "Precision": precision_score(y_test, candidate_pred, pos_label=1),
            "Recall": recall_score(y_test, candidate_pred, pos_label=1),
            "F1-score": f1_score(y_test, candidate_pred, pos_label=1),
            "AUC": model_auc,
        }
    )
    for curve_fpr, curve_tpr in zip(fpr, tpr):
        roc_rows.append(
            {
                "Model": model_name,
                "False Positive Rate": curve_fpr,
                "True Positive Rate": curve_tpr,
                "AUC": model_auc,
            }
        )

roc_rows.extend(
    [
        {"Model": "Random Classifier", "False Positive Rate": 0.0, "True Positive Rate": 0.0, "AUC": 0.5},
        {"Model": "Random Classifier", "False Positive Rate": 1.0, "True Positive Rate": 1.0, "AUC": 0.5},
    ]
)

roc_curve_data = pd.DataFrame(roc_rows)
model_comparison = pd.DataFrame(model_comparison_rows).sort_values("AUC", ascending=False)

#Perhitungan Metrik Evaluasi
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label=1),
    "recall": recall_score(y_test, y_pred, pos_label=1),
    "f1_score": f1_score(y_test, y_pred, pos_label=1),
    "auc": auc(*roc_curve(y_test, y_proba, pos_label=1)[:2]),
    "train_accuracy": accuracy_score(y_train, y_train_pred),
    "generalization_gap": accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_pred),
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=[1, 0]),
    "test_actual": y_test,
    "test_pred": y_pred,
    "test_proba": y_proba,
    "train_size": len(x_train),
    "train_size_smote": len(y_train_smote),
    "test_size": len(x_test),
}

#Menampilkan Hasil
model_comparison.reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1-score,AUC
0,Naive Bayes,0.9750,1.0,0.96,0.979592,1.0
1,SVM,0.9875,1.0,0.98,0.989899,1.0
2,Random Forest,1.0000,1.0,1.00,1.000000,1.0


## 7. Hasil 5-Fold Cross Validation

Bagian ini menguji kestabilan model Random Forest memakai Stratified 5-Fold Cross Validation. Data dibagi menjadi 5 bagian; setiap bagian bergantian menjadi data uji, sementara 4 bagian lainnya menjadi data latih. SMOTE ikut dijalankan di dalam pipeline sehingga hanya diterapkan pada data latih setiap fold.

In [8]:
# Cell ini menghitung dan menampilkan hasil 5-Fold Cross Validation beserta rata-rata akurasinya.

cv = StratifiedKFold(n_splits=9, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    create_model_pipeline(model_type="random_forest", n_estimators=250),
    x,
    y,
    cv=cv,
    scoring="accuracy",
)

cv_results = pd.DataFrame(
    {
        "Fold": [f"Fold {i + 1}" for i in range(len(cv_scores))],
        "Akurasi": cv_scores.round(4),
        "Akurasi (%)": (cv_scores * 100).round(2),
    }
)

metrics["cv_mean"] = float(cv_scores.mean())
metrics["cv_std"] = float(cv_scores.std())
metrics["cv_scores"] = cv_scores

print("Hasil Dari 9-Fold Cross Validation")
display(cv_results)

print("\nHasil Rata-Rata Akurasi 9-Fold Cross Validation")
display(
    pd.DataFrame(
        [
            {
                "Rata-Rata Akurasi": round(float(cv_scores.mean()), 4),
                "Rata-Rata Akurasi (%)": round(float(cv_scores.mean()) * 100, 2),
                "Standar Deviasi (%)": round(float(cv_scores.std()) * 100, 2),
            }
        ]
    )
)

Hasil Dari 9-Fold Cross Validation


,Fold,Akurasi,Akurasi (%)
0,Fold 1,0.9778,97.78
1,Fold 2,1.0000,100.00
2,Fold 3,1.0000,100.00
3,Fold 4,1.0000,100.00
4,Fold 5,1.0000,100.00
5,Fold 6,0.9773,97.73
6,Fold 7,1.0000,100.00
7,Fold 8,1.0000,100.00
8,Fold 9,1.0000,100.00



Hasil Rata-Rata Akurasi 9-Fold Cross Validation


,Rata-Rata Akurasi,Rata-Rata Akurasi (%),Standar Deviasi (%)
0,0.995,99.5,0.93


## 8. Membuat Grafik Akurasi Random Forest

Bagian ini mencoba beberapa jumlah pohon pada Random Forest. Nilai akurasi dari setiap jumlah pohon disimpan agar nanti bisa ditampilkan sebagai grafik di aplikasi Streamlit.

In [9]:
# Cell ini membandingkan akurasi Random Forest dengan beberapa jumlah pohon.
# Hasilnya dipakai untuk grafik akurasi di aplikasi Streamlit.

accuracy_rows = []
for n_estimators in [25, 50, 100, 150, 200, 250]:
    candidate = create_model_pipeline(n_estimators=n_estimators)
    scores = cross_val_score(candidate, x, y, cv=cv, scoring="accuracy")
    accuracy_rows.append(
        {
            "Jumlah Pohon": n_estimators,
            "Akurasi": scores.mean() * 100,
            "Std": scores.std() * 100,
        }
    )

accuracy_curve = pd.DataFrame(accuracy_rows)
accuracy_curve

,Jumlah Pohon,Akurasi,Std
0,25,99.001122,1.547683
1,50,99.500561,0.934440
2,100,99.500561,0.934440
3,150,99.500561,0.934440
4,200,99.500561,0.934440
5,250,99.500561,0.934440


## 9. ROC Curve per Kelas untuk Model Random Forest

Bagian ini menghitung ROC Curve untuk **masing-masing kelas** pada model Random Forest, yaitu kelas CKD dan kelas Not CKD, ditambah garis acuan Random Classifier (AUC = 0.5000). Data kurva ini disimpan ke artifact dan ditampilkan sebagai grafik di aplikasi Streamlit.

In [10]:
# Cell ini menyiapkan data ROC Curve per kelas untuk model Random Forest.
# Hasilnya dipakai untuk grafik "ROC Curves per Class for Random Forest Model" di Streamlit.

rf_proba_all = eval_pipeline.predict_proba(x_test)

roc_per_class_rows = []
for class_index, class_name in CLASS_LABELS.items():
    class_fpr, class_tpr, _ = roc_curve(y_test, rf_proba_all[:, class_index], pos_label=class_index)
    class_auc = auc(class_fpr, class_tpr)
    for curve_fpr, curve_tpr in zip(class_fpr, class_tpr):
        roc_per_class_rows.append(
            {
                "Kelas": class_name,
                "False Positive Rate": curve_fpr,
                "True Positive Rate": curve_tpr,
                "AUC": class_auc,
            }
        )

roc_per_class_rows.extend(
    [
        {"Kelas": "Random Classifier", "False Positive Rate": 0.0, "True Positive Rate": 0.0, "AUC": 0.5},
        {"Kelas": "Random Classifier", "False Positive Rate": 1.0, "True Positive Rate": 1.0, "AUC": 0.5},
    ]
)

roc_per_class_data = pd.DataFrame(roc_per_class_rows)

print("Ringkasan AUC per Kelas untuk Model Random Forest")
display(roc_per_class_data.groupby("Kelas", as_index=False)["AUC"].max())

Ringkasan AUC per Kelas untuk Model Random Forest


,Kelas,AUC
0,CKD,1.0
1,Not CKD,1.0
2,Random Classifier,0.5


## 10. Training Model Final dan Feature Importance

Bagian ini melatih model final memakai seluruh data yang tersedia. Model final inilah yang dipakai oleh aplikasi. Setelah model dilatih, nilai feature importance dihitung untuk mengetahui fitur mana yang paling berpengaruh pada prediksi.

In [11]:
# Cell ini melatih model final dengan seluruh data yang sudah dibersihkan.
# SMOTE tetap berjalan di dalam pipeline saat proses fit.
# Setelah itu, feature importance dihitung untuk melihat fitur paling berpengaruh.

#Pembangunan Model Final
production_pipeline = create_model_pipeline(n_estimators=250)
production_pipeline.fit(x, y)

preprocessor = production_pipeline.named_steps["preprocessor"]
forest = production_pipeline.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

importance_rows = []
for name, importance in zip(feature_names, forest.feature_importances_):
    raw_name = name.split("__", 1)[-1]
    source_feature = raw_name.split("_", 1)[0] if name.startswith("cat__") else raw_name
    importance_rows.append({"Fitur": source_feature, "Importance": importance})

feature_importance = pd.DataFrame(importance_rows)
feature_importance = feature_importance.groupby("Fitur", as_index=False)["Importance"].sum()
feature_importance["Persentase"] = feature_importance["Importance"] / feature_importance["Importance"].sum() * 100
feature_importance["Nama Fitur"] = feature_importance["Fitur"].map(FIELD_LABELS).fillna(feature_importance["Fitur"])
feature_importance = feature_importance.sort_values("Persentase", ascending=False)
feature_importance.head(10)

,Fitur,Importance,Persentase,Nama Fitur
10,hemo,0.186566,18.656600,Hemoglobin
14,pcv,0.182133,18.213286,Packed cell volume
19,sc,0.127126,12.712602,Serum creatinine
20,sg,0.124303,12.430306,Specific gravity
11,htn,0.082700,8.270007,Hypertension
1,al,0.064065,6.406533,Albumin
9,dm,0.057585,5.758546,Diabetes mellitus
18,rc,0.055992,5.599224,Red blood cell count
7,bu,0.025104,2.510434,Blood urea
5,bgr,0.022283,2.228320,Blood glucose random


## 11. Menyimpan Model dan Semua Artifact

Bagian terakhir ini menyimpan model final, metrik evaluasi, data grafik akurasi, data ROC Curve, data ROC Curve per kelas, feature importance, dan nilai default form ke `model_artifacts.joblib`. File ini wajib ada agar `app.py` bisa menjalankan prediksi tanpa training ulang.

In [12]:
# Cell ini menggabungkan model final, metrik evaluasi, grafik akurasi,
# ROC Curve, ROC Curve per kelas, feature importance, dan nilai default form ke dalam satu file artifact.
# File model_artifacts.joblib inilah yang akan dibaca oleh app.py.

artifacts = {
    "model": production_pipeline,
    "metrics": metrics,
    "accuracy_curve": accuracy_curve,
    "roc_curve_data": roc_curve_data,
    "roc_per_class_data": roc_per_class_data,
    "model_comparison": model_comparison,
    "cv_results": cv_results,
    "feature_importance": feature_importance,
    "numeric_defaults": df[NUMERIC_COLUMNS].median(numeric_only=True).to_dict(),
    "categorical_defaults": {
        col: df[col].mode(dropna=True).iloc[0]
        for col in CATEGORICAL_COLUMNS
        if not df[col].mode(dropna=True).empty
    },
    "features": FEATURES,
}

joblib.dump(artifacts, MODEL_FILE)
print(f"Model dan metrik tersimpan ke: {MODEL_FILE}")

Model dan metrik tersimpan ke: c:\Users\ASUS\OneDrive\Documents\22. joki 12 mei (terbaru)\model_artifacts.joblib
